# Bài 4.3 · Fine-tuning trên ảnh thật

**Oxford-IIIT Pet** có 7 349 ảnh chụp chó mèo thuộc 37 giống, chia sẵn khoảng 3 680 ảnh train và 3 669 ảnh test. Ảnh chụp thật: nền lộn xộn, tư thế đủ kiểu, nhiều giống nhìn rất giống nhau.

Notebook so hai cách dùng cùng một ResNet-50 đã học trên ImageNet:

1. **Chỉ học head**: đóng băng toàn bộ backbone, kể cả BatchNorm, chỉ học lớp cuối.
2. **Fine-tune cả mạng**: mở toàn bộ, backbone học với learning rate nhỏ hơn head mười lần.

Cuối notebook xem các ảnh bị đoán sai, rồi lưu và tải lại model ở một bản mới để kiểm logits khớp.

Cần bật **GPU** (T4 hoặc P100) và **Internet** trong *Session options*. Chạy hết khoảng 10 phút.

In [ ]:
import time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader

random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| thiết bị:',
      torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
EPOCHS, BATCH = 3, 64

## 1. Dữ liệu

Ảnh train được cắt ngẫu nhiên và lật ngang để mô hình thấy nhiều biến thể; ảnh test chỉ thu về 256 rồi cắt giữa 224. Hai phép chuẩn hóa lấy đúng mean, std mà ResNet-50 dùng khi học trên ImageNet: lệch chuẩn hóa là lệch phân phối đầu vào của model pretrained.

In [ ]:
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
mean, std = weights.transforms().mean, weights.transforms().std
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(mean, std)])
test_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(mean, std)])

root = '/tmp/pets'
train_ds = torchvision.datasets.OxfordIIITPet(root, split='trainval', transform=train_tf, download=True)
test_ds = torchvision.datasets.OxfordIIITPet(root, split='test', transform=test_tf, download=True)
view_ds = torchvision.datasets.OxfordIIITPet(root, split='test', transform=T.Compose([T.Resize(256), T.CenterCrop(224)]))
classes = train_ds.classes
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
test_dl = DataLoader(test_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True)
print(len(train_ds), 'ảnh train |', len(test_ds), 'ảnh test |', len(classes), 'lớp')

In [ ]:
idx = random.sample(range(len(view_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, i in zip(axes.flat, idx):
    img, y = view_ds[i]
    ax.imshow(img); ax.set_title(classes[y], fontsize=10); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Một vòng train và một vòng chấm

Dùng chung cho cả hai cách. Huấn luyện với số thực 16 bit (`autocast`) để nhanh hơn trên GPU; `GradScaler` nhân loss trước backward để gradient nhỏ không bị làm tròn về 0.

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

def run_epoch(model, loader, opt=None, sched=None):
    train = opt is not None
    total = correct = 0; loss_sum = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.set_grad_enabled(train), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
            logits = model(x)
            loss = nn.functional.cross_entropy(logits, y)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None:
                sched.step()
        loss_sum += loss.item() * len(y); correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum / total, correct / total

def new_model():
    m = torchvision.models.resnet50(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, len(classes))
    return m.to(device)

count = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

## 3. Chỉ học head

`requires_grad = False` chặn gradient tới backbone, nhưng **chưa đủ**: BatchNorm ở chế độ train vẫn cập nhật trung bình chạy theo từng batch. Vì vậy mỗi epoch đặt cả mạng về `eval()` rồi chỉ bật `train()` cho lớp `fc`.

In [ ]:
head = new_model()
for p in head.parameters():
    p.requires_grad = False
for p in head.fc.parameters():
    p.requires_grad = True
opt = torch.optim.AdamW(head.fc.parameters(), lr=1e-3)
print(f'tham số được học: {count(head):,} trên {sum(p.numel() for p in head.parameters()):,}')

hist = []
for ep in range(EPOCHS):
    t = time.time()
    head.eval(); head.fc.train()
    tl, ta = run_epoch(head, train_dl, opt)
    head.eval(); vl, va = run_epoch(head, test_dl)
    hist.append(('chỉ head', ep + 1, ta, va))
    print(f'epoch {ep + 1}: train acc {ta:.3f} | test acc {va:.3f} | {time.time() - t:.0f} giây')

## 4. Fine-tune cả mạng

Mở toàn bộ tham số. Backbone đã biết nhìn cạnh, lông, mắt nên chỉ cần chỉnh nhẹ: learning rate 1e-4, còn head mới học từ đầu nên dùng 1e-3. `OneCycleLR` tăng dần rồi giảm learning rate trong suốt quá trình.

In [ ]:
full = new_model()
groups = [{'params': [p for n, p in full.named_parameters() if not n.startswith('fc.')], 'lr': 1e-4},
          {'params': full.fc.parameters(), 'lr': 1e-3}]
opt = torch.optim.AdamW(groups, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[1e-4, 1e-3], total_steps=EPOCHS * len(train_dl))
print(f'tham số được học: {count(full):,}')

for ep in range(EPOCHS):
    t = time.time()
    full.train(); tl, ta = run_epoch(full, train_dl, opt, sched)
    full.eval(); vl, va = run_epoch(full, test_dl)
    hist.append(('cả mạng', ep + 1, ta, va))
    print(f'epoch {ep + 1}: train acc {ta:.3f} | test acc {va:.3f} | {time.time() - t:.0f} giây')

In [ ]:
h = pd.DataFrame(hist, columns=['cách', 'epoch', 'train acc', 'test acc'])
fig, ax = plt.subplots(figsize=(6, 4))
for name, g in h.groupby('cách'):
    ax.plot(g['epoch'], g['test acc'], marker='o', label=name)
ax.set_xlabel('epoch'); ax.set_ylabel('test accuracy'); ax.set_xticks(range(1, EPOCHS + 1)); ax.legend(); ax.grid(alpha=.3)
plt.show()
h.pivot(index='epoch', columns='cách', values='test acc').round(3)

## 5. Ảnh bị đoán sai

Điểm tổng chưa nói model sai ở đâu. Xem vài ảnh sai thường thấy ngay lỗi nằm ở các giống trông gần giống nhau, hay ở ảnh mà chính người cũng khó đoán.

In [ ]:
full.eval(); preds, labels = [], []
with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
    for x, y in test_dl:
        preds.append(full(x.to(device)).argmax(1).cpu()); labels.append(y)
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
wrong = np.flatnonzero(preds != labels)
print(f'sai {len(wrong)} / {len(labels)} ảnh')
fig, axes = plt.subplots(2, 4, figsize=(12, 6.5))
for ax, i in zip(axes.flat, np.random.default_rng(0).choice(wrong, 8, replace=False)):
    img, y = view_ds[i]
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'thật: {classes[y]}\nđoán: {classes[preds[i]]}', fontsize=9)
plt.tight_layout(); plt.show()
pairs = pd.Series([f'{classes[labels[i]]} → {classes[preds[i]]}' for i in wrong]).value_counts().head(8)
pairs

## 6. Lưu rồi tải lại ở một bản mới

Bài nộp chỉ đáng tin khi model nạp lại từ tệp cho ra đúng logits như lúc vừa train. Tạo một mạng mới, nạp `state_dict`, đặt `eval()`, rồi so trên cùng một batch.

In [ ]:
torch.save(full.state_dict(), '/tmp/pets_resnet50.pt')
again = new_model(); again.load_state_dict(torch.load('/tmp/pets_resnet50.pt', map_location=device)); again.eval()
x, _ = next(iter(test_dl)); x = x.to(device)
with torch.no_grad():
    diff = (full(x) - again(x)).abs().max().item()
print('chênh logits lớn nhất:', diff)

## Đọc kết quả

- Chỉ học head đã cho độ chính xác cao ngay từ epoch đầu: đặc trưng ImageNet đủ tốt để tách phần lớn các giống. Fine-tune cả mạng nhích thêm vài điểm vì backbone được chỉnh theo chi tiết lông, tai, mõm của 37 giống.
- Số tham số được học chênh nhau khoảng 300 lần (khoảng 76 nghìn so với 23,6 triệu), nên chỉ học head là lựa chọn tốt khi dữ liệu ít hoặc thời gian gấp.
- Các cặp sai nhiều nhất thường là những giống trông gần giống nhau; muốn cải thiện thì nhìn vào chính các cặp này trước khi đổi kiến trúc.

Gợi ý khi demo: bỏ dòng `head.eval(); head.fc.train()` để BatchNorm của backbone chạy ở chế độ train, chạy lại mục 3 và so độ chính xác test.